# Contested Norms Study
## Data Prep: Create derived fields

Created: 2025-10-13  
Authors: Edward L. Platt  

In [ ]:
%matplotlib inline
from configparser import ConfigParser
import csv
from datetime import datetime, timedelta
import json
import logging
import math
import os
import pytz
import simplejson as json
import sys
from tqdm import tqdm

utc=pytz.UTC

### LOAD CONFIGURATION
config = ConfigParser()
config.read('config.ini')
config.write(sys.stdout)

subreddit_id = config.get("Source", "subreddit_id")
start_time = utc.localize(datetime.strptime(config.get("Source", "start_time"), "%Y-%m-%d %H:%M:%S"))
end_time = utc.localize(datetime.strptime(config.get("Source", "end_time"), "%Y-%m-%d %H:%M:%S"))

account_file = config.get("Transformed", "account_file")

script_name = "contested_norms-{}-sample".format(subreddit_id)
script_date = datetime.now().strftime('%Y-%m-%d')

# Configure logging
logging.basicConfig(
    filename='{}-{}.log'.format(script_date, script_name),
    format='%(asctime)s:%(levelname)s:%(message)s',
    level=logging.DEBUG)
logger = logging.getLogger("CivilServant-Analysis")
handler = logging.StreamHandler(sys.stdout)
handler.setLevel(logging.DEBUG)
formatter = logging.Formatter('%(asctime)s:%(levelname)s:%(message)s')
handler.setFormatter(formatter)
logger.addHandler(handler)

### LOAD ANALYSIS CODE
from pipeline import DataFile
from pipeline.transforms import CivilServantTransformToAccounts

In [ ]:
transform = CivilServantTransformToAccounts(
    start_time, end_time, subreddit_id)
accounts = DataFile(transform, directory="transformed", filename=account_file, load=True)

In [ ]:
import pandas as pd
df = pd.DataFrame(accounts.rows())
for c in df.columns:
    print(c)
    df[c] = df[c].apply(lambda value: None if value == "" else value).astype(dtype[c])

In [ ]:
num_accounts = len(df)
deciles = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]
df_percentiles = pd.DataFrame({
    'num.accounts': [math.floor(x * num_accounts) for x in deciles],
    'num.comments': df['num.comments'].quantile(deciles),
    'num.nonspam.comments': (df['num.comments'] - df['num.comments.spam']).quantile(deciles),
    'num.posts': df['num.posts'].quantile(deciles),
    'num.nonspam.posts': (df['num.posts'] - df['num.posts.spam']).quantile(deciles),
    'percentile': range(10, 110, 10),
}).set_index('percentile')
df_percentiles

In [ ]:
df_noban = df[df['num.bans'] == 0]

In [ ]:
num_accounts = len(df_noban)
deciles = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]
df_percentiles = pd.DataFrame({
    'num.accounts': [math.floor(x * num_accounts) for x in deciles],
    'num.comments': df_noban['num.comments'].quantile(deciles),
    'num.nonspam.comments': (df_noban['num.comments'] - df_noban['num.comments.spam']).quantile(deciles),
    'num.posts': df_noban['num.posts'].quantile(deciles),
    'num.nonspam.posts': (df_noban['num.posts'] - df_noban['num.posts.spam']).quantile(deciles),
    'percentile': range(10, 110, 10),
}).set_index('percentile')
df_percentiles